In [11]:
import requests
from datetime import datetime

def get_historical_weather(city: str, match_date: datetime) -> dict:
    # get coordinates for the city
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {"name": f"{city}, United Kingdom", "count": 1}
    geo_json = requests.get(geo_url, params=geo_params).json()
    
    lat = geo_json["results"][0]["latitude"]
    lon = geo_json["results"][0]["longitude"]
    date_str = match_date.strftime("%Y-%m-%d")
    
    # get weather for that date
    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_str,
        "end_date": date_str,
        "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max",
        "timezone": "Europe/London"
    }
    
    r = requests.get(weather_url, params=weather_params).json()
    
    return {
        "city": city,
        "match_date": date_str,
        "temp_min_c": r["daily"]["temperature_2m_min"][0],
        "temp_max_c": r["daily"]["temperature_2m_max"][0],
        "precipitation_mm": r["daily"]["precipitation_sum"][0],
        "windspeed_kmh": r["daily"]["windspeed_10m_max"][0],
    }

In [12]:
print(get_historical_weather('Liverpool', pd.to_datetime('03/01/2023')))


{'city': 'Liverpool', 'match_date': '2023-03-01', 'temp_min_c': 3.8, 'temp_max_c': 8.4, 'precipitation_mm': 0.7, 'windspeed_kmh': 19.2}


In [18]:
season_year = '2025'
url = "https://v3.football.api-sports.io/standings"
headers = {
    'x-rapidapi-key': 'c07c641a0ab474bd2d9a789a88637210',
    'x-rapidapi-host': 'v3.football.api-sports.io'
}
params = {
    'league': '39',
    'season': str(season_year)
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

In [19]:
data

{'get': 'standings',
 'parameters': {'league': '39', 'season': '2025'},
 'errors': {'plan': 'Free plans do not have access to this season, try from 2022 to 2024.'},
 'results': 0,
 'paging': {'current': 1, 'total': 1},
 'response': []}

In [ ]:
if not data.get('response'):
    print(f"no data for {season_year}")
    return []

standings = data['response'][0]['league']['standings'][0]

team_list = []
for entry in standings:
    team_data = entry['team']
    team_list.append({
        'team': team_data['name'],
        'id': team_data['id']
    })

return team_list

teams_2018 = get_teams_api_football(2018)
print(f"Found {len(teams_2018)} teams.")
print(teams_2018)



In [24]:
def get_weather(location, match_date):
    date_str = match_date.strftime("%Y-%m-%d")

    if not location:
        logger.warning(f"no location provided for match on {date_str}")
        return {
            "location": None,
            "match_date": date_str,
            "temp_min": None,
            "temp_max": None,
            "precipitation_mm": None,
            "windspeed_kmh": None,
        }
    
    # get coordinates for the venue
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_params = {"name": f'{location}, England', "count": 1}
    geo_json = requests.get(geo_url, params=geo_params).json()

    # if no results from geocoding
    if not geo_json.get("results"):
        logger.warning(f"no coordinates found for venue: {location}")
        return {
            "location": location,
            "match_date": date_str,
            "temp_min": None,
            "temp_max": None,
            "precipitation_mm": None,
            "windspeed_kmh": None,
        }
        
    lat = geo_json["results"][0]["latitude"]
    lon = geo_json["results"][0]["longitude"]
        
    # get weather for that date
    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_str,
        "end_date": date_str,
        "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max",
        "timezone": "Europe/London"
    }
        
    r = requests.get(weather_url, params=weather_params).json()
    
    return {
        "location": location,
        "match_date": date_str,
        "temp_min": r["daily"]["temperature_2m_min"][0],
        "temp_max": r["daily"]["temperature_2m_max"][0],
        "precipitation_mm": r["daily"]["precipitation_sum"][0],
        "windspeed_kmh": r["daily"]["windspeed_10m_max"][0],
    }

In [25]:
venues = ["Anfield", "Goodison Park"]
for venue in venues:
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_json = requests.get(geo_url, params={"name": f'{venue}, United Kingdom', "count": 1}).json()
    print(f"{venue}: {geo_json}")

Anfield: {'generationtime_ms': 1.3991594}
Goodison Park: {'generationtime_ms': 1.850605}


In [26]:
def get_weather(lat, lon, match_date):
    date_str = match_date.strftime("%Y-%m-%d")

    if not lat and not lon:
        logger.warning(f"no location provided for match on {date_str}")
        return {
            "location": None,
            "match_date": date_str,
            "temp_min": None,
            "temp_max": None,
            "precipitation_mm": None,
            "windspeed_kmh": None,
        }
            
    # get weather for that date
    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_str,
        "end_date": date_str,
        "daily": "temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max",
        "timezone": "Europe/London"
    }
        
    r = requests.get(weather_url, params=weather_params).json()
    return r
    return {
        "coordinates": (lat, lon),
        "match_date": date_str,
        "temp_min": r["daily"]["temperature_2m_min"][0],
        "temp_max": r["daily"]["temperature_2m_max"][0],
        "precipitation_mm": r["daily"]["precipitation_sum"][0],
        "windspeed_kmh": r["daily"]["windspeed_10m_max"][0],
    }
    

In [27]:
print(get_weather(53.788799,-2.23018, pd.to_datetime('01/01/2023')))

{'latitude': 53.813705, 'longitude': -2.2543335, 'generationtime_ms': 8.50975513458252, 'utc_offset_seconds': 3600, 'timezone': 'Europe/London', 'timezone_abbreviation': 'GMT+1', 'elevation': 121.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_min': '°C', 'temperature_2m_max': '°C', 'precipitation_sum': 'mm', 'windspeed_10m_max': 'km/h'}, 'daily': {'time': ['2023-01-01'], 'temperature_2m_min': [5.8], 'temperature_2m_max': [7.8], 'precipitation_sum': [10.4], 'windspeed_10m_max': [29.0]}}


In [ ]:
# # rename for clarity
    # results = results.rename(columns={
    # 'FTHG': 'FT Home Goals',
    # 'FTAG': 'FT Away Goals',
    # 'FTR': 'FT Result',
    # 'HTHG': 'HT Home Goals',
    # 'HTAG': 'HT Away Goals',
    # 'HTR': 'HT Result',
    # 'HS': 'Home Shots',
    # 'AS': 'Away Shots',
    # 'HST': 'Home SoT',
    # 'AST': 'Away SoT',
    # 'HC': 'Home Corners',
    # 'AC': 'Away Corners',
    # 'HF': 'Home Fouls',
    # 'AF': 'Away Fouls',
    # 'HY': 'Home Yellows',
    # 'AY': 'Away Yellows',
    # 'HR': 'Home Reds',
    # 'AR': 'Away Reds'
    # })

In [5]:
import pandas as pd
from pathlib import Path

notebook_dir = Path.cwd() 
root_dir = notebook_dir.parent

historical_seasons = ['1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425']
for season in historical_seasons:
    source_path = root_dir / "data" / "bronze" / "weather_data" / f"season_{season}.csv"
    df = pd.read_csv(source_path)
    df[['latitude', 'longitude']] = df['coordinates'].str.strip('()').str.split(',', expand=True).astype(float)
    df = df.drop(columns=['coordinates'])
    df.to_csv(source_path, index=False)

In [8]:
# scripts/fix_coordinates.py
import pandas as pd
from pathlib import Path

seasons = ['1516', '1617', '1718', '1819', '1920', '2021', '2122', '2223', '2324', '2425']

notebook_dir = Path.cwd() 
root_dir = notebook_dir.parent

TEAMS_TO_FIX = ['Man United', 'Man City']

for season in seasons:
    weather_path = root_dir / "data" / "bronze" / "weather_data" / f"season_{season}.csv"
    fixtures_path = root_dir / "data" / "silver" / "cleaned_fb_data" / f"season_{season}.csv"

    weather_df = pd.read_csv(weather_path)
    fixtures_df = pd.read_csv(fixtures_path)

    # get the correct coordinates from fixtures for affected teams
    affected = fixtures_df[fixtures_df['home_team'].isin(TEAMS_TO_FIX)][['match_date', 'home_team', 'latitude', 'longitude']]

    for _, row in affected.iterrows():
        # find the weather row for this date and update its coordinates
        mask = weather_df['match_date'] == row['match_date']
        weather_df.loc[mask, 'latitude'] = row['latitude']
        weather_df.loc[mask, 'longitude'] = row['longitude']
        print(f"fixed {row['home_team']} on {row['match_date']}")

    weather_df.to_csv(weather_path, index=False)
    print(f"saved {season}")

fixed Man United on 2015-08-08
fixed Man City on 2015-08-16
fixed Man United on 2015-08-22
fixed Man City on 2015-08-29
fixed Man United on 2015-09-12
fixed Man City on 2015-09-19
fixed Man United on 2015-09-26
fixed Man City on 2015-10-03
fixed Man City on 2015-10-17
fixed Man United on 2015-10-25
fixed Man City on 2015-10-31
fixed Man United on 2015-11-07
fixed Man City on 2015-11-21
fixed Man City on 2015-11-28
fixed Man United on 2015-12-05
fixed Man City on 2015-12-12
fixed Man United on 2015-12-19
fixed Man City on 2015-12-26
fixed Man United on 2015-12-28
fixed Man United on 2016-01-02
fixed Man City on 2016-01-13
fixed Man City on 2016-01-16
fixed Man United on 2016-01-23
fixed Man United on 2016-02-02
fixed Man City on 2016-02-06
fixed Man City on 2016-02-14
fixed Man United on 2016-02-28
fixed Man United on 2016-03-02
fixed Man City on 2016-03-05
fixed Man City on 2016-03-20
fixed Man United on 2016-04-03
fixed Man City on 2016-04-09
fixed Man United on 2016-04-16
fixed Man U